# 06 · Where does the model look?

A clinician will not trust a model that is right for the wrong reasons. This
notebook asks where on the ECG the phase 3 baseline looks when it detects each
superclass, and whether the answer is about the model at all.

Three tools, all in `fedecg.explain.saliency`:

- **Integrated Gradients** (Sundararajan et al., 2017) credits every sample of
  every lead with part of the prediction, relative to a baseline input.
- **Grad-CAM** (Selvaraju et al., 2017) weights the last residual stage's
  activations by their gradient: a coarser map over time, shared by all leads.
- **Beat delineation** with neurokit2 splits lead II into QRS complex, ST
  segment and T wave. *Enrichment* is the share of attribution in a segment
  divided by the share of time it covers: 1 is chance.

Each is checked against the **model-randomization test** (Adebayo et al.,
2018): the same maps from the same network with random weights. A map that
looks the same either way shows the input, not the model.

Prerequisite:

```bash
uv run python scripts/explain_model.py
```

## A trap: the zero baseline

Integrated Gradients is usually run from an all-zero input. That fails here.
The ResNet's first convolution has no bias and is followed by GroupNorm, so the
network is **scale-invariant**: f(a·x) = f(x) for every a > 0. The path from
zero to x is a rescaling of x, so the output does not change along it until
it jumps at exactly 0.

In [ ]:
import pandas as pd
import torch

from fedecg.models.resnet1d import build_model
from fedecg.paths import CHECKPOINT_DIR, FIGURES_DIR, TABLES_DIR

state = torch.load(CHECKPOINT_DIR / "centralized.pt", weights_only=False, map_location="cpu")
model = build_model(state["config"]["model"]).eval()
model.load_state_dict(state["model_state"])
x = torch.randn(1, 12, 1000)
with torch.no_grad():
    for a in [1.0, 0.1, 0.01, 0.0]:
        print(f"a = {a:<5} probabilities {torch.sigmoid(model(a * x))[0].numpy().round(3)}")

The attributions here therefore start from a **blurred copy of each record**
(Gaussian, 0.2 s), which removes the QRS, ST and T morphology but keeps the
slow trend (Sturmfels et al., 2020). They answer: what does the *shape* of the
waveform contribute?

## Does each map depend on the model?

Median Spearman correlation between maps of the trained and the random-weight
network, over 100 correctly detected test ECGs per class. The last column is
the share of records whose random-weight Grad-CAM is not all zero (after its
ReLU, an untrained network's map is often empty).

In [ ]:
pd.read_csv(TABLES_DIR / "saliency_sanity.csv", index_col="superclass")

## Where the attention goes

Enrichment per segment for the trained model and its random-weight twin.

In [ ]:
segments = pd.read_csv(TABLES_DIR / "saliency_segments.csv")
for method in ["integrated_gradients", "grad_cam"]:
    table = segments[segments["method"].isin([method, f"{method}_random"])]
    display(
        table.pivot(index="superclass", columns=["segment", "method"], values="enrichment")
        .round(2)
        .style.set_caption(method)
    )

## Which leads

Share of Integrated Gradients attribution per lead, minus the share for
normal ECGs. Positive means the class draws more attention to that lead.

In [ ]:
leads = pd.read_csv(TABLES_DIR / "saliency_leads.csv", index_col="superclass")
(leads - leads.loc["NORM"]).drop(index="NORM").round(3)

## One example per class

In [ ]:
from IPython.display import Image, display

for name in ["MI", "STTC", "HYP"]:
    display(Image(filename=FIGURES_DIR / f"saliency_{name}.png"))

## What this means

These results were computed twice, for the 32-channel baseline before tuning
and for the tuned 64-channel one. What held both times is stated as a finding;
what changed is flagged.

- **Zero-baseline Integrated Gradients is invalid for this network.** It is
  scale-invariant, so the integration path carries no information. Any
  architecture that normalizes right after a bias-free first layer has the
  same property.
- **Even with a blurred baseline, IG mostly shows the input** (held for both
  networks). Its maps correlate at ρ ≈ 0.5 with those of an untrained network,
  and both put 2.3 to 3× their share of attribution on the QRS complex, where
  the signal differs most from its blurred copy. IG heatmaps of ECGs should
  not be read on their own; only the difference from the random network is
  informative.
- **Grad-CAM mostly depends on the model**: ρ between −0.22 and 0.01 with the
  random network for MI, CD and HYP. For STTC (0.29) and, in the tuned
  network, NORM (0.51) it is partly the input's shape, so those two classes
  are read with care.
  - **MI is recognized from the QRS complex, not the ST segment** (held for
    both networks: QRS 2.9× and 2.7×, ST 0.5× and 0.7×). Most PTB-XL
    infarctions are old ones, whose lasting sign is a pathological Q wave in
    the QRS; acute ST elevation is rare in this dataset. So the answer to "ST
    segment or noise?" is neither: QRS morphology.
  - **Conduction disturbances draw learned attention to the QRS (2.24× vs.
    0.49× for the random network) and the early ST segment (2.80× vs. 0.07×)**
    (held for both networks). A widened QRS can spill past the boundary the
    delineator draws, so part of that "ST" attention may be the QRS tail.
  - **Hypertrophy changed between networks.** The smaller network's attention
    was learned mostly on the ST segment (1.55× vs. 0.06×); the tuned one's is
    learned mostly on the QRS (1.70× vs. 0.37×), where the voltage criteria
    for hypertrophy live, with less on the ST segment (1.31× vs. 0.70×). Both
    are clinically plausible; which one a model uses is not stable across
    trainings.
  - **STTC** puts extra attention on the T wave (1.63× vs. 1.35× for the random
    network); its ST emphasis (1.62×) is no higher than the random network's.
- **Lead shifts need a control too.** By lead, HYP puts extra attribution on
  V5 and V1, the leads of the Sokolow-Lyon voltage criterion (held for both
  networks). MI's extra attribution in III, aVL and V1 looks like an inferior
  and septal pattern, but conduction disturbances show a similar shift, so it
  is not specific to infarction.